# Reproducing Krishnan, Heer, Cherian & Sheffield (2022) from dandiset 000462

Paper. Krishnan S, Heer C, Cherian C, Sheffield MEJ (2022). Reward expectation extinction restructures and degrades CA1 spatial maps through loss of a dopaminergic reward proximity signal. *Nature Communications* 13, 6662. [doi:10.1038/s41467-022-34465-5](https://doi.org/10.1038/s41467-022-34465-5)

Dataset. [DANDI:000462](https://dandiarchive.org/dandiset/000462/0.230416.2132), version 0.230416.2132 ([doi:10.48324/dandi.000462/0.230416.2132](https://doi.org/10.48324/dandi.000462/0.230416.2132)), streamed. The paper cites no version of the dandiset. This is the only published version; the draft (modified 2023-06-20) has the same 14 assets and size and is not fixed, so the published version is pinned.

Claims. `claims/000462/10.1038-s41467-022-34465-5.yaml`, written from the version of record (PMC9636178) with its Supplementary Information, and checked by a second reader.

What the dandiset holds. One file per mouse: 10 wild-type mice of the reward extinction experiment (5 with R, UR, RR and a novel environment; 5 with R and UR only), and 4 DAT-Cre hM4D(Gi) mice with day 1 (R, UR, RR) and days 2 to 4 (before and after saline, CNO and DCZ). Each file holds raw suite2p fluorescence and three behaviour channels at 30 Hz. It does not hold the time-control mice, the dark sessions, the pupil video, the tdTomato control mice or the VTA axon imaging, and it has 10 of the 12 wild-type mice and 4 of the 6 DREADD mice.

| Claim | Tested here |
|---|---|
| session-lap-counts, lick-persistence-ur | tested |
| fig1d, fig1e, fig1f, fig1g, supp4-lickstop-definition-robust, supp4-lapwise-r2, supp5-not-running-speed | tested (decoder) |
| approach-ratio (R and UR; the dark sessions are not in the dandiset), engaged-lap-counts, supp6b-decoder-engaged-laps, disengaged-laps-uniform | tested |
| fig2a, fig2c, fig2d, fig2e, fig3a, fig3b (three claims), fig3c, supp11, fig3d, reward-zone-field-turnover | tested (place fields) |
| supp13-novel-environment | tested |
| supp16, fig4f, supp18, fig4h, fig4m | tested (DREADD mice) |
| supp2-decoder-stable-control, controls-no-change-over-time | blocked: the control mice are not in the dandiset |
| supp6-pupil-engagement | blocked: no pupil data |
| supp15-tdtomato-controls | blocked: the tdTomato mice are not in the dandiset |
| fig5-da-axon-ramps, fig5k-ramp-slope-max, fig5l-ramps-return-rr, supp20-novel-axons, supp21-heterogeneous-axons | blocked: no axon imaging |
| supp14-reward-cells | blocked: the reward cell criteria are in ref. 24, not in the paper |

The analysis of every mouse (dF/F, transients, decoders, place fields) runs once, in the next cells, with the helper `krishnan.py`; each claim's section then reads the results. Where each parameter comes from is in the helper's docstring.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import krishnan as K

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
WT = K.WT_FULL + K.WT_RUR
DR = K.DREADD
print("dandiset", K.DANDISET, "version", K.VERSION)

def mean_ci(x):
    """Mean and half-width of the t 95% confidence interval."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return (float(x.mean()) if len(x) else np.nan, np.nan)
    return float(x.mean()), float(stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

In [ ]:
t0 = time.time()
RES = K.process_all(WT + DR, workers=2)
print(f"processed {len(RES)} mice in {(time.time() - t0) / 60:.1f} min")

## Data check

In [ ]:
assets = K.assets()
print("assets:", len(assets), "| subjects:", sorted(assets))
rows = []
for s, r in RES.items():
    rows.append({"mouse": s, "description": r["subject_description"][:40], "cells": r["n_cells"], "fluorescence frames": r["f_shape"][0],
                 "behaviour rows in file": r["behaviour_rows_in_file"], "F min": round(r["F_min"]), "F median": round(r["F_median"]),
                 "offset removed": round(r["offset"]), "position V": tuple(round(v, 3) for v in r["volts"]["position_V"]),
                 "200 cm at V": round(r["scale"], 3), "reward V": tuple(round(v, 2) for v in r["volts"]["reward_V"]),
                 "lick V": tuple(round(v, 2) for v in r["volts"]["lick_V"]),
                 "reward between 0.5-4 V": round(r["volts"]["reward_frac_between_0.5_and_4V"], 4),
                 "lick between 0.5-4 V": round(r["volts"]["lick_frac_between_0.5_and_4V"], 4),
                 "behaviour md5": r["fingerprints"]["behaviour_md5"], "fluorescence md5": r["fingerprints"]["fluorescence_md5"]})
info = pd.DataFrame(rows).set_index("mouse")
print(info.to_string())
print("distinct behaviour fingerprints:", info["behaviour md5"].nunique(), "| distinct fluorescence fingerprints:", info["fluorescence md5"].nunique(), "of", len(info))
cond_rows = []
for s, r in RES.items():
    for n, c in r["beh"]["conds"].items():
        cond_rows.append({"mouse": s, "condition": n, "frames": c["n_frames"], "complete laps": len(K.complete_laps(c)),
                          "lap segments": len(c["laps"]), "rewards": len(c["rewards"]), "lick onsets": len(c["licks"]),
                          "transient frac": round(r["transient_frac"][n], 3), "dF/F 1/50/99 pct": tuple(round(v, 3) for v in r["dff_pct"][n])})
cdf = pd.DataFrame(cond_rows)
print(cdf.to_string(index=False))
print("speed range (cm/s) over running frames:", round(min(np.percentile(c["speed"][c["speed"] >= 1], 1) for r in RES.values() for c in r["beh"]["conds"].values()), 1),
      "to", round(max(np.percentile(c["speed"], 99.9) for r in RES.values() for c in r["beh"]["conds"].values()), 1))

The checks against the description:

- Frame counts: the description of each fluorescence series lists the conditions and their frame counts; they sum to the fluorescence frames in every file. The behaviour rows match the fluorescence frames in 11 files; sub-NR6 has 8 extra behaviour rows at the end, sub-CFC4 one row fewer, and sub-014 6000 rows fewer (next cell).
- Conditions: rewards are delivered in R, RR, the novel session and all DREADD sessions, and not in UR, consistent with the condition order in the description, except that in 4 mice (NR6, NR14, NR23, CFC16) one reward is delivered at the end of the first UR lap (the `rewards` column above): either the boundary taken from the frame counts is one lap early or the reward was switched off one lap late. The lap is kept in UR, as the frame counts say.
- Units: the position channel is in volts (0 to about 0.61 V) with no stated scale; the reward is delivered at the top of the range, so the top is read as 200 cm (assumption). The reward and lick channels are two-level signals (almost no samples between 0.5 and 4 V).
- Fluorescence: the wild-type files have a minimum near 0 counts; the DREADD files have a minimum near 17000 counts with transients a few thousand counts above it, an offset the description does not mention. It is removed (file minimum) before dF/F.
- The 14 files are distinct: no two share the behaviour or fluorescence fingerprints.
- The 30 Hz rate is the series' stated rate; the description gives no units for position, reward or licks.

In [ ]:
# Clock check: decoding R position with the true labels against labels shifted by 10 s
for s in WT:
    d = RES[s]["dec_shift"]
    print(f"{s}: held-out R2 with aligned labels {d[0]:.3f}, with labels shifted 10 s {d[300]:.3f}")
# Locating the missing behaviour frames of sub-014
gap = pd.DataFrame(K.gap_check("014"))
print(gap.round(1).to_string(index=False))

In every wild-type mouse the position decodes from the fluorescence with aligned labels (R2 above 0.7) and not with labels shifted by 10 s, so behaviour and fluorescence share a clock at the scale of the decoder. In sub-014 the fluorescence after frame 71000 matches the behaviour 6000 frames earlier; before it, it matches with no shift (table above, a model trained on the aligned BeforeSaline condition). The 6000 missing behaviour rows are therefore at the start of BeforeCNO; the helper inserts them as missing (`BEHAVIOUR_GAPS`), and laps that touch them are dropped. For the DREADD mice the clock is shown by the place fields found in every condition (next sections); no decoder was run on them.

## session-lap-counts
> Mice ran on average 34 laps in R, 28 in UR and 27 in RR (mean +/- 95% CI 34 +/- 2, 28 +/- 4 and 27 +/- 3), and 31 +/- 5 laps in the novel rewarded environment.

Complete laps (start in the first 10 cm, reach the last 5 cm) and all lap segments, which include a partial lap at the start or end of a condition. The code variant drops the frames the paper's code drops (`task_framestokeep`).

In [ ]:
def code_kept_laps(s, cond_i, c):
    keep = K.CODE_FRAMES_TO_KEEP[s][cond_i]
    stop = keep if keep > 0 else c["n_frames"] + keep
    return sum(1 for a, e, ok in c["laps"] if ok and e <= stop)

rows = []
for s in WT:
    conds = RES[s]["beh"]["conds"]
    row = {"mouse": s}
    for i, (n, c) in enumerate(conds.items()):
        row[n] = len(K.complete_laps(c))
        row[n + " (with partial)"] = len(c["laps"])
        row[n + " (code frames)"] = code_kept_laps(s, i, c)
    rows.append(row)
laps_df = pd.DataFrame(rows).set_index("mouse")
print(laps_df.T.to_string())
expect = {"Rewarded": (34, 2), "Unrewarded": (28, 4), "ReRewarded": (27, 3), "Novel": (31, 5)}
lapcount_measured = {}
for n, (v, tol) in expect.items():
    for suffix in ["", " (with partial)", " (code frames)"]:
        m, h = mean_ci(laps_df[n + suffix].dropna())
        ok = abs(m - v) <= tol
        if suffix == "":
            lapcount_measured[n] = m
        print(f"{n + suffix:32s} mean {m:5.1f} +/- {h:4.1f} (n = {laps_df[n + suffix].notna().sum()}); paper {v} +/- {tol}; within: {ok}")

**Result: reproduced.** Counting complete laps, the means are 35.3 (R, 10 mice), 29.7 (UR), 28.2 (RR, 5 mice) and 32.8 (novel, 5 mice), all within the paper's 34 +/- 2, 28 +/- 4, 27 +/- 3 and 31 +/- 5. Counting partial laps puts R at 36.8, outside its tolerance; the code's frame cut gives 33.2.

## lick-persistence-ur
> After reward removal mice kept pre-licking for about 5 laps (5 +/- 1) before stopping; 4 of 12 mice licked randomly in later laps.

In [ ]:
print("reward zone start per mouse (cm, mean first R lick position after 20 cm):",
      {s: round(RES[s]["beh"]["zone"], 1) for s in WT})
print("code variant zone start (cm):", {s: RES[s]["beh"]["zone_code"] for s in WT})
lick_rows = []
for s in WT:
    b = RES[s]["beh"]
    z = b["ur_zone_licks"]
    anyl = b["ur_any_licks"]
    first_nolick = int(np.argmax(z == 0)) if (z == 0).any() else len(z)
    row = {"mouse": s, "UR laps": len(z), "zone licks per UR lap": " ".join(map(str, z))}
    for L in K.LICK_READINGS:
        ls = b["lick_stop"][L]
        row[f"RE high laps {L}"] = ls
        row[f"(a) zone lick in RE low {L}"] = bool((z[ls:] > 0).any())
        row[f"(b) any lick in RE low {L}"] = bool((anyl[ls:] > 0).any())
    row["(c) zone lick after first no-lick lap"] = bool((z[first_nolick + 1:] > 0).any())
    lick_rows.append(row)
lick_df = pd.DataFrame(lick_rows).set_index("mouse")
print(lick_df.drop(columns="zone licks per UR lap").to_string())
for s in WT:
    print(s, "zone licks per UR lap:", lick_df.loc[s, "zone licks per UR lap"])
lick_persist = {}
for L in K.LICK_READINGS:
    m, h = mean_ci(lick_df[f"RE high laps {L}"])
    lick_persist[L] = m
    print(f"{L}: RE high laps per mouse mean {m:.2f} +/- {h:.2f} (95% CI, n = {len(WT)}); paper 5 +/- 1; within: {abs(m - 5) <= 1}")
    print(f"    random lickers: (a) {lick_df[f'(a) zone lick in RE low {L}'].sum()}, (b) {lick_df[f'(b) any lick in RE low {L}'].sum()} of {len(WT)} mice")
print(f"    (c) {lick_df['(c) zone lick after first no-lick lap'].sum()} of {len(WT)} mice")
# code variant of the zone
for L in K.LICK_READINGS:
    v = [K.lick_stop(RES[s]["beh"]["ur_zone_licks_code"] > 0, 2, L) for s in WT]
    print(f"code zone variant {L}: RE high laps mean {np.mean(v):.2f}, per mouse {v}")

**Result: partially reproduced.** Mice kept licking in the reward zone for several UR laps, but the mean number of RE high laps is 6.60 +/- 2.62 under L1 (7.60 under L2, 8.60 under L3), outside 5 +/- 1 under every reading. Random licking after the lick stop by reading (a), a zone lick in an RE low lap, is found in 4 of the 10 mice, the paper's count; reading (b) gives 7 and reading (c) 9.

## fig1f-decoder-r2
> Mean decoder R2 is 0.95 in R, 0.90 in RE high laps (similar to R), 0.65 in RE low laps (significantly lower), and 0.82 in RR, which recovers somewhat but stays below R.

Decoder: Gaussian naive Bayes on the dF/F of all cells, running frames, 40 bins, trained on the first 60% of R laps, the track treated as a circle (each prediction taken at its nearest copy of the true bin), R2 per lap between true and predicted bin (Methods, Position decoding, as the criterion gives it). The same decoder without the circular treatment is shown for comparison. Per mouse (main reading, from the code's `plot_shuffleerror_withlickstop`): R is the mean over random draws of 4 of the last 5 R laps, RE high and RE low of draws of min(4, lick stop) and min(3, lick stop) laps from those laps, RR of min(4, lick stop) RR laps. The mean over many such draws converges to the mean over the laps drawn from, which is computed exactly here (last 5 R laps, all RE high, all RE low, all RR laps). The variant is the code's decoder (first 50% of R frames, unwrapped).

In [ ]:
def lap_r2(s, cond, key="dec"):
    return {d["lap"]: d["r2"] for d in RES[s][key][cond]}

def per_mouse_r2(s, L, key="dec", reading="main"):
    b = RES[s]["beh"]
    ls = b["lick_stop"][L]
    r = lap_r2(s, "Rewarded", key)
    u = lap_r2(s, "Unrewarded", key)
    out = {}
    rl = [r[k] for k in sorted(r)]
    out["R"] = np.nanmean(rl[-5:]) if reading == "main" else np.nanmean(rl)
    out["RE high"] = np.nanmean([v for k, v in u.items() if k < ls]) if any(k < ls for k in u) else np.nan
    out["RE low"] = np.nanmean([v for k, v in u.items() if k >= ls]) if any(k >= ls for k in u) else np.nan
    if "ReRewarded" in RES[s][key]:
        out["RR"] = np.nanmean(list(lap_r2(s, "ReRewarded", key).values()))
    else:
        out["RR"] = np.nan
    return out

def paired(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    return stats.ttest_rel(a[ok], b[ok]), ok.sum()

fig1f = {}
for key, label in [("dec", "Methods decoder (60% of R laps, circular)"), ("dec_var", "code decoder (50% of R frames)"), ("dec_lin", "60% of R laps, not circular")]:
    for reading in ["main", "all tested R laps"]:
        for L in K.LICK_READINGS:
            df = pd.DataFrame({s: per_mouse_r2(s, L, key, reading) for s in WT}).T
            means = {c: mean_ci(df[c].dropna()) for c in df.columns}
            tests = {}
            for a, b2 in [("R", "RE high"), ("R", "RE low"), ("R", "RR"), ("RE high", "RE low"), ("RE high", "RR"), ("RE low", "RR")]:
                t, n = paired(df[a], df[b2])
                tests[(a, b2)] = (t.statistic, t.pvalue, n)
            fig1f[(key, reading, L)] = (df, means, tests)
            print(f"--- {label}, R laps {reading}, lick stop {L}")
            print("  ", "; ".join(f"{c} {m:.3f} [{m - h:.3f} {m + h:.3f}] (n={df[c].notna().sum()})" for c, (m, h) in means.items()))
            print("  ", "; ".join(f"{a}-{b2}: t={t:.2f} p={p:.4f} (x6 Bonferroni {min(1, 6 * p):.4f}, n={n})" for (a, b2), (t, p, n) in tests.items()))
df, means, tests = fig1f[("dec", "main", "L1")]
print("\nper mouse, Methods decoder, main reading, L1:\n", df.round(3).to_string())
fig1f_ok = {}
for k, (df, means, tests) in fig1f.items():
    if k[0] == "dec_lin":
        continue
    within = all(abs(means[c][0] - v) <= tol for c, (v, tol) in {"R": (0.95, 0.02), "RE high": (0.90, 0.03), "RE low": (0.65, 0.11), "RR": (0.82, 0.05)}.items())
    for corr in [1, 6]:
        sig = (tests[("R", "RE low")][1] * corr < 0.05, tests[("R", "RR")][1] * corr < 0.05, tests[("R", "RE high")][1] * corr >= 0.05)
        fig1f_ok[k + (corr,)] = (within, sig)
        print(k, "Bonferroni x6" if corr == 6 else "uncorrected", "means within tolerance:", within, "| R<RE low, R<RR significant, R vs RE high not:", sig)

**Result: partially reproduced.** Under lick stop L1 the per-mouse means are R 0.960, RE high 0.901, RE low 0.606 and RR 0.861, all within tolerance (also under L2). RE low is below R (p = 0.0008, 0.0048 with Bonferroni) and RR below R (p = 0.0345 uncorrected, 0.207 with Bonferroni). RE high is not similar to R: the paired t-test gives p = 0.0047 (0.028 with Bonferroni), where the criterion requires p >= 0.05; this holds under every reading and with the code's decoder.

## fig1d-decoder-drop-ur
> A naive Bayes decoder trained on initial R laps predicts position accurately on the final R laps and on the first UR laps, then abruptly underperforms later in UR.

Per mouse: holds when the mean R2 of the first 3 UR laps is within 0.1 of the mean R2 of the held-out R laps and the mean R2 of the RE low laps is below both (criterion; thresholds the reader's).

In [ ]:
for s in WT:
    r = lap_r2(s, "Rewarded")
    u = lap_r2(s, "Unrewarded")
    print(s, "held-out R:", " ".join(f"{r[k]:.2f}" for k in sorted(r)))
    print(s, "UR        :", " ".join(f"{u[k]:.2f}" for k in sorted(u)))
fig1d = {}
for key in ["dec", "dec_var", "dec_lin"]:
    for L in K.LICK_READINGS:
        hold = 0
        rows = []
        for s in WT:
            r = lap_r2(s, "Rewarded", key)
            u = lap_r2(s, "Unrewarded", key)
            ls = RES[s]["beh"]["lick_stop"][L]
            rmean = np.nanmean(list(r.values()))
            first3 = np.nanmean([u[k] for k in sorted(u)[:3]])
            low = np.nanmean([v for k, v in u.items() if k >= ls])
            ok = abs(first3 - rmean) <= 0.1 and low < first3 and low < rmean
            hold += ok
            rows.append((s, round(rmean, 3), round(first3, 3), round(low, 3), ok))
        fig1d[(key, L)] = hold
        print(f"{key} {L}: holds in {hold} of {len(WT)} mice:", rows)

**Result: reproduced.** With the decoder of the Methods (60% of R laps, circular track), the first 3 UR laps decode within 0.1 of the held-out R laps and RE low laps decode worse than both in 8 of the 10 mice under each lick stop reading; the criterion's 9 of 12 is 75% of the mice, and 8 of 10 is 80%. The code's decoder gives 6 of 10. Without the circular treatment, 2 of 10.

## fig1e-decoder-follows-licking
> Averaged over 12 mice, lap by lap decoder R2 in UR falls as an S-shaped curve around the lap where pre-licking reaches zero; a reverse Boltzmann sigmoid fits the mean R2 with r = 0.94, and its point of maximum change is at lap 10.

In [ ]:
from scipy.optimize import curve_fit

def boltz(x, top, bottom, v50, slope):
    return bottom + (top - bottom) / (1 + np.exp((x - v50) / slope))

def roll3(x):
    return pd.Series(x).rolling(3, min_periods=1).mean().to_numpy()

NL = 15
r2_mat = np.full((len(WT), NL), np.nan)
lk_mat = np.full((len(WT), NL), np.nan)
for i, s in enumerate(WT):
    u = lap_r2(s, "Unrewarded")
    z = RES[s]["beh"]["ur_zone_licks"]
    arr = np.array([u.get(k, np.nan) for k in range(len(z))])
    sm = roll3(arr)[:NL]
    r2_mat[i, :len(sm)] = sm
    lz = roll3(z.astype(float))[:NL]
    lk_mat[i, :len(lz)] = lz
mean_r2 = np.nanmean(r2_mat, axis=0)
mean_lk = np.nanmean(lk_mat, axis=0)
x = np.arange(1, NL + 1)
p, _ = curve_fit(boltz, x, mean_r2, p0=[mean_r2[:3].mean(), mean_r2[-3:].mean(), 8, 1], maxfev=20000)
fit = boltz(x, *p)
r_fit = np.corrcoef(fit, mean_r2)[0, 1]
R2_fit = 1 - ((mean_r2 - fit) ** 2).sum() / ((mean_r2 - mean_r2.mean()) ** 2).sum()
xf = np.linspace(1, NL, 2001)
d2 = np.gradient(np.gradient(boltz(xf, *p), xf), xf)
zc = np.where(np.diff(np.sign(d2)) != 0)[0]
lap_max = xf[zc[0]] if len(zc) else np.nan
lk_norm = mean_lk / np.nanmax(mean_lk)
lk_min_lap = int(x[np.argmax(lk_norm <= lk_norm.min() + 1e-9)])
print("mean R2 per UR lap (3-lap rolling, mice present):", np.round(mean_r2, 3))
print("mice per lap:", np.isfinite(r2_mat).sum(0))
print("mean zone licks per lap (rolling):", np.round(mean_lk, 2))
print(f"sigmoid params top {p[0]:.3f} bottom {p[1]:.3f} v50 {p[2]:.2f} slope {p[3]:.2f}")
print(f"fit r = {r_fit:.3f}, fit R2 = {R2_fit:.3f} (paper 0.94 +/- 0.094)")
print(f"first zero of second derivative at lap {lap_max:.2f} (rounded {np.round(lap_max):.0f}; paper 10 +/- 1)")
print(f"normalised lick curve first reaches its minimum at lap {lk_min_lap}; distance to the lap of maximum change {abs(lk_min_lap - np.round(lap_max)):.0f}")

**Result: reproduced.** The reverse Boltzmann fit to the mean UR R2 has r = 0.979 (R2 of the fit 0.959; paper 0.94), its second derivative first reaches zero at lap 11.23, rounded to 11 (paper 10 +/- 1), and the smoothed lick curve first reaches its minimum at lap 14, 3 laps from it, within the 3 laps the criterion allows.

## fig1g-decoder-error-all-locations
> In RE low laps decoder error increases at all locations along the track (start, middle and end), not only near the reward; in RR the error decreases across the track.

Error per frame = distance in cm between true and decoded position (circular), averaged per mouse over the frames of each condition in the start (0 to 50 cm), middle (50 to 150 cm) and end (150 to 200 cm) groups.

In [ ]:
GROUPS = {"start": (0, 10), "middle": (10, 30), "end": (30, 40)}

def group_error(s, cond, laps_sel, key="dec"):
    out = {}
    recs = [d for d in RES[s][key][cond] if laps_sel(d["lap"])]
    if not recs:
        return {g: np.nan for g in GROUPS}
    bins = np.concatenate([d["bins"] for d in recs])
    err = np.concatenate([d["err_cm"] for d in recs])
    for g, (a, b) in GROUPS.items():
        m = (bins >= a) & (bins < b)
        out[g] = err[m].mean() if m.any() else np.nan
    return out

fig1g = {}
for key in ["dec", "dec_var"]:
    for L in K.LICK_READINGS:
        rows = {}
        for s in WT:
            ls = RES[s]["beh"]["lick_stop"][L]
            rows[(s, "R")] = group_error(s, "Rewarded", lambda l: True, key)
            rows[(s, "RE low")] = group_error(s, "Unrewarded", lambda l, ls=ls: l >= ls, key)
            if "ReRewarded" in RES[s][key]:
                rows[(s, "RR")] = group_error(s, "ReRewarded", lambda l: True, key)
        df = pd.DataFrame(rows).T
        res_g = {}
        for g in GROUPS:
            r = df.xs("R", level=1)[g]
            lo = df.xs("RE low", level=1)[g].reindex(r.index)
            t = stats.ttest_rel(lo, r)
            rr = df.xs("RR", level=1)[g]
            t2 = stats.ttest_rel(rr, r.reindex(rr.index))
            res_g[g] = (r.mean(), lo.mean(), rr.mean(), t.statistic, t.pvalue, t2.statistic, t2.pvalue)
        fig1g[(key, L)] = res_g
        print(f"--- {key} {L}")
        for g, v in res_g.items():
            print(f"  {g:6s}: R {v[0]:.1f} cm, RE low {v[1]:.1f} cm, RR {v[2]:.1f} cm (5 mice); RE low vs R t={v[3]:.2f} p={v[4]:.4f}; RR vs R t={v[5]:.2f} p={v[6]:.4f}")
        n_sig = sum(v[1] > v[0] and v[4] < 0.01 for v in res_g.values())
        print(f"  groups with RE low error higher at p < 0.01: {n_sig} of 3")
print("error range over mice and groups (cm):", round(float(np.nanmin(df.values)), 1), "to", round(float(np.nanmax(df.values)), 1))

**Result: reproduced.** Under L1, decoder error in RE low is higher than in R at the start (17.1 against 4.9 cm, p < 0.0001), middle (33.9 against 5.6 cm, p = 0.0001) and end (17.3 against 4.4 cm, p < 0.0001) of the track, in 3 of 3 groups under every lick stop reading and with the code's decoder. RR errors (12.1, 21.0 and 11.6 cm in the 5 mice) lie between R and RE low.

## supp4-lickstop-definition-robust
> The drop in decoder R2 from RE high to RE low holds whether the lick stop is the first no-lick lap or after 2 or 4 consecutive no-lick laps, or when laps are grouped by presence of licks; with 6 consecutive no-lick laps the result differs. Median R2 is 0.96 in R and 0.80 in RR, and about 0.87 to 0.93 in RE high and 0.69 to 0.70 in RE low across definitions.

In [ ]:
def r2_split(s, ls=None, lick_mask=None, key="dec"):
    u = lap_r2(s, "Unrewarded", key)
    if lick_mask is not None:
        hi = [v for k, v in u.items() if lick_mask[k]]
        lo = [v for k, v in u.items() if not lick_mask[k]]
    else:
        hi = [v for k, v in u.items() if k < ls]
        lo = [v for k, v in u.items() if k >= ls]
    return (np.nanmean(hi) if hi else np.nan), (np.nanmean(lo) if lo else np.nan)

supp4 = {}
for start in ["first", "Nth", "after"]:
    rows = []
    for name, n in [("first no-lick lap", 1), ("2 laps", 2), ("4 laps", 4), ("6 laps", 6), ("presence of licks", None)]:
        his, los = [], []
        for s in WT:
            z = RES[s]["beh"]["ur_zone_licks"]
            if n is None:
                hi, lo = r2_split(s, lick_mask=z > 0)
            else:
                L = {"first": "L1", "Nth": "L2", "after": "L3"}[start]
                ls = K.lick_stop(z > 0, n, L)
                hi, lo = r2_split(s, ls)
            his.append(hi)
            los.append(lo)
        his, los = np.array(his), np.array(los)
        ok = np.isfinite(his) & np.isfinite(los)
        t = stats.ttest_rel(his[ok], los[ok])
        rows.append({"definition": name, "n mice": int(ok.sum()), "median RE high": np.nanmedian(his), "median RE low": np.nanmedian(los),
                     "t": t.statistic, "p": t.pvalue, "p x5 Bonferroni": min(1, 5 * t.pvalue)})
    df = pd.DataFrame(rows)
    supp4[start] = df
    print(f"--- RE low starts at the {start} lap of the run")
    print(df.round(4).to_string(index=False))
medR = np.median([per_mouse_r2(s, "L1")["R"] for s in WT])
medRR = np.nanmedian([per_mouse_r2(s, "L1")["RR"] for s in WT])
print(f"median R {medR:.3f} (paper 0.96 +/- 0.096), median RR {medRR:.3f} (paper 0.80 +/- 0.08)")
for start, df in supp4.items():
    d = df[df.definition != "6 laps"]
    ok_t = (d["p x5 Bonferroni"] < 0.05).all() and (d["median RE high"] > d["median RE low"]).all()
    ok_hi = d["median RE high"].between(0.87 - 0.087, 0.93 + 0.093).all()
    ok_lo = d["median RE low"].between(0.69 - 0.069, 0.70 + 0.07).all()
    print(f"{start}: tests significant for first/2/4/presence: {ok_t}; RE high medians in 0.783 to 1.023: {ok_hi}; RE low medians in 0.621 to 0.77: {ok_lo}")

**Result: reproduced.** With RE low starting at the first lap of the run (the first start reading), RE low R2 is below RE high for the first no-lick lap, 2 and 4 laps and the grouping by licks (Bonferroni p 0.006 to 0.012), RE high medians are 0.91 and RE low medians 0.66 to 0.67, inside the widened ranges, and the median R (0.976) and RR (0.842) are within tolerance. Under the other two start readings the 4-lap test is not significant after correction (p = 0.066 and 0.065). The 6-lap definition gives p = 0.30 (Nth reading) and 0.12 (after reading) after correction; under the first reading it is the same split as 2 and 4 laps.

## supp4-lapwise-r2
> Over all laps pooled, decoder R2 is 0.97 (SD 0.046) in R (399 laps), 0.86 (0.14) in RE high (90 laps), 0.56 (0.34) in RE low (244 laps) and 0.79 (0.19) in RR (121 laps).

In [ ]:
for L in K.LICK_READINGS:
    pools = {"R held-out": [], "R all laps (incl. training)": [], "RE high": [], "RE low": [], "RR": []}
    for s in WT:
        ls = RES[s]["beh"]["lick_stop"][L]
        pools["R held-out"] += list(lap_r2(s, "Rewarded").values())
        pools["R all laps (incl. training)"] += list(lap_r2(s, "Rewarded").values()) + list(lap_r2(s, "Rewarded_train").values())
        u = lap_r2(s, "Unrewarded")
        pools["RE high"] += [v for k, v in u.items() if k < ls]
        pools["RE low"] += [v for k, v in u.items() if k >= ls]
        if "ReRewarded" in RES[s]["dec"]:
            pools["RR"] += list(lap_r2(s, "ReRewarded").values())
    print(f"--- {L}: " + "; ".join(f"{k}: {len(v)} laps, mean {np.nanmean(v):.3f} (SD {np.nanstd(v, ddof=1):.3f})" for k, v in pools.items()))
    print("    R2 range over pooled laps:", round(min(np.nanmin(v) for v in pools.values()), 2), "to", round(max(np.nanmax(v) for v in pools.values()), 2))
print("total complete R laps:", sum(len(K.complete_laps(RES[s]['beh']['conds']['Rewarded'])) for s in WT))

**Result: partially reproduced.** The pooled lap means are within the stated SDs under every reading (L3: R 0.977 held out or 0.990 with training laps, RE high 0.858, RE low 0.635, RR 0.869). The lap counts are not: R has 353 complete laps (paper 399), RE low 211 to 231 (244 +/- 24 needs 220 or more; L1 gives 231 but then RE high has 66 against 90 +/- 9), and RR 141 (121 +/- 12). The dandiset holds 10 of the 12 mice.

## supp5-not-running-speed
> The drop in decoder performance in UR is not due to running; 70% of UR laps (233 of 334) have lap times matched to R, and decoder R2 falls in matched laps as in slower laps (median R 0.96, matched 0.76, slower 0.71). Matched laps take 7.34 s in R and 7.4 s in UR, slower laps 20 s.

In [ ]:
def running_lap_times(c):
    """Code variant (GetBehaviorStats_withGoodrunning.py, good_lap_time): running frames only."""
    return np.array([(c["speed"][s:e] >= K.RUN_SPEED).sum() / K.FS for s, e in K.complete_laps(c)])

for reading in ["R min to max", "R 5th to 95th percentile", "R min to max, running time only (code)"]:
    n_m = n_s = 0
    tm_r, tm_u, ts_u = [], [], []
    med = {"R": [], "matched": [], "slower": []}
    for s in WT:
        lt = RES[s]["beh"]["lap_times"]
        r, u = lt["Rewarded"], lt["Unrewarded"]
        if "code" in reading:
            r = running_lap_times(RES[s]["beh"]["conds"]["Rewarded"])
            u = running_lap_times(RES[s]["beh"]["conds"]["Unrewarded"])
        lo, hi = (np.nanmin(r), np.nanmax(r)) if reading.startswith("R min") else tuple(np.nanpercentile(r, [5, 95]))
        matched = (u >= lo) & (u <= hi)
        n_m += matched.sum()
        n_s += (~matched).sum()
        tm_r += list(r)
        tm_u += list(u[matched])
        ts_u += list(u[~matched])
        ur = lap_r2(s, "Unrewarded")
        med["R"].append(np.nanmean(list(lap_r2(s, "Rewarded").values())))
        med["matched"].append(np.nanmean([ur[k] for k in ur if matched[k]]) if any(matched[k] for k in ur) else np.nan)
        med["slower"].append(np.nanmean([ur[k] for k in ur if not matched[k]]) if any(not matched[k] for k in ur) else np.nan)
    w = stats.wilcoxon(np.array(med["R"]), np.array(med["matched"]))
    print(f"--- {reading}: matched {n_m}, slower {n_s} of {n_m + n_s} UR laps ({100 * n_m / (n_m + n_s):.0f}% matched)")
    print(f"    lap time R {np.nanmean(tm_r):.2f} s, matched UR {np.nanmean(tm_u):.2f} s, slower UR {np.nanmean(ts_u):.2f} s")
    print(f"    median over mice of per-mouse mean R2: R {np.nanmedian(med['R']):.3f}, matched {np.nanmedian(med['matched']):.3f}, slower {np.nanmedian(med['slower']):.3f}")
    print(f"    R vs matched Wilcoxon W={w.statistic:.1f} p={w.pvalue:.4f} (x3 Bonferroni {min(1, 3 * w.pvalue):.4f}), n = {len(WT)} mice")

**Result: partially reproduced.** Decoder R2 falls in UR laps matched to R in lap time as it does in slower laps (median over mice R 0.985, matched 0.763, slower 0.695, all within tolerance; R against matched Wilcoxon p = 0.0059 after Bonferroni). The counts and times are not reproduced: 172 matched and 125 slower laps of 297 (paper 233 and 101), and lap times of 12.23 s in R and 13.38 s in matched UR laps (paper 7.34 and 7.4 s). Counting running time only, as the code does, gives 203 and 94 laps and 8.50 s and 9.16 s, still outside.

## approach-ratio
> Mice slow down approaching the track end; the middle to end velocity ratio is 1.01 in the dark, 1.3 in R and 1.22 in UR, so approach behaviour is an engagement readout.

The dark sessions are not in the dandiset; R and UR are tested. Ratio per lap = mean binned velocity at 100 to 150 cm over 175 to 200 cm.

In [ ]:
appr = {}
for key, label in [("ratios", "Methods ratio, whole lap"), ("ratios_trav", "Methods ratio, traversal only"), ("ratios_code", "code ratio, whole lap"), ("ratios_code_trav", "code ratio, traversal only")]:
    for cond in ["Rewarded", "Unrewarded"]:
        pooled = np.concatenate([RES[s]["beh"][key][cond] for s in WT])
        per_mouse = [np.nanmean(RES[s]["beh"][key][cond]) for s in WT]
        m1, h1 = mean_ci(pooled)
        m2, h2 = mean_ci(per_mouse)
        appr[(key, cond)] = (m1, h1, m2, h2)
        print(f"{label:30s} {cond:10s}: over laps {m1:.3f} +/- {h1:.3f} (n = {np.isfinite(pooled).sum()} laps); over mice {m2:.3f} +/- {h2:.3f} (n = {len(WT)})")
        print(f"{'':30s} {'':10s}  range over laps {np.nanmin(pooled):.2f} to {np.nanmax(pooled):.2f}")
for s in WT:
    print(s, "R", round(np.nanmean(RES[s]["beh"]["ratios"]["Rewarded"]), 3), "UR", round(np.nanmean(RES[s]["beh"]["ratios"]["Unrewarded"]), 3))

**Result: not reproduced.** Mice slow down approaching the end in both conditions and more in R than in UR, but the ratio of the Methods is 5.17 in R and 4.44 in UR over whole laps (2.21 and 1.75 over the traversal only), against 1.3 and 1.22. The code's ratio over the traversal is closest, 1.259 and 1.199, both outside the tolerances of 0.02. The dark sessions are not in the dandiset.

## engaged-lap-counts
> Approach behaviour stays on most UR laps after licking stops; 170 of 244 RE low laps (70%) are engaged and 74 (30%) disengaged, with 90 RE high laps.

In [ ]:
eng_counts = {}
for reading, ekey, rkey in [("whole lap", "engaged_ur", "ratios"), ("traversal only", "engaged_ur_trav", "ratios_trav")]:
  print("=== approach ratio over the", reading)
  for L in K.LICK_READINGS:
      n_hi = n_eng = n_dis = 0
      r_eng, r_dis = [], []
      per = []
      for s in WT:
          b = RES[s]["beh"]
          ls = b["lick_stop"][L]
          e = b[ekey]
          rat = b[rkey]["Unrewarded"]
          n_hi += ls
          n_eng += e[ls:].sum()
          n_dis += (~e[ls:]).sum()
          r_eng += list(rat[ls:][e[ls:]])
          r_dis += list(rat[ls:][~e[ls:]])
          per.append((s, ls, int(e[ls:].sum()), int((~e[ls:]).sum()), round(b["engage_thr" if ekey == "engaged_ur" else "engage_thr_trav"], 3)))
      eng_counts[(reading, L)] = (n_eng, n_dis, n_hi, np.nanmean(r_eng), np.nanmean(r_dis))
      tot = n_eng + n_dis
      print(f"{L}: RE high {n_hi}, RE low {tot}: engaged {n_eng} ({100 * n_eng / tot:.0f}%), disengaged {n_dis}; "
            f"ratio engaged {np.nanmean(r_eng):.3f} +/- {mean_ci(r_eng)[1]:.3f}, disengaged {np.nanmean(r_dis):.3f} +/- {mean_ci(r_dis)[1]:.3f}")
      print("    per mouse (mouse, RE high laps, engaged, disengaged, threshold):", per)

**Result: partially reproduced.** With the whole-lap ratio, 71% of RE low laps are engaged under every lick stop reading (paper 70%); under L1 the engaged (163) and disengaged (68) counts are within tolerance but the RE high laps (66) are not; under L3 the RE high count (86) is within it and the engaged count (151) is not. No reading puts all three counts in tolerance, in 10 of the 12 mice. The mean ratio of engaged laps (4.41) is far from 1.3. With the traversal-only ratio only 44 to 46% of laps are engaged.

## supp6b-decoder-engaged-laps
> Decoder R2 is as low in engaged RE low laps (0.65) as in all RE low laps, while disengaged laps reduce it further (0.31).

In [ ]:
for L in K.LICK_READINGS:
    rows = []
    for s in WT:
        b = RES[s]["beh"]
        ls = b["lick_stop"][L]
        e = b["engaged_ur"]
        u = lap_r2(s, "Unrewarded")
        eng = [v for k, v in u.items() if k >= ls and e[k]]
        dis = [v for k, v in u.items() if k >= ls and not e[k]]
        rows.append({"mouse": s, "R": per_mouse_r2(s, L)["R"], "RE low": per_mouse_r2(s, L)["RE low"],
                     "engaged": np.nanmean(eng) if eng else np.nan, "disengaged": np.nanmean(dis) if dis else np.nan,
                     "n engaged": len(eng), "n disengaged": len(dis)})
    df = pd.DataFrame(rows).set_index("mouse")
    me, he = mean_ci(df["engaged"])
    md, hd = mean_ci(df["disengaged"])
    ok = df[["R", "engaged"]].dropna()
    w = stats.wilcoxon(ok["R"], ok["engaged"])
    ok2 = df[["engaged", "disengaged"]].dropna()
    print(f"--- {L}: engaged {me:.3f} [{me - he:.3f} {me + he:.3f}] (n={df['engaged'].notna().sum()}), disengaged {md:.3f} [{md - hd:.3f} {md + hd:.3f}] (n={df['disengaged'].notna().sum()}); "
          f"R vs engaged Wilcoxon p={w.pvalue:.4f}; mice with both: {len(ok2)}, engaged > disengaged in {(ok2['engaged'] > ok2['disengaged']).sum()}")
    print(df.round(3).to_string())

**Result: reproduced.** Under L1, engaged RE low laps decode at R2 0.686 [0.565 0.808] (paper 0.65) and disengaged laps at 0.415 [0.015 0.816] (paper 0.31), both within tolerance; engaged laps are below R (Wilcoxon p = 0.0020) and disengaged laps are below engaged on average, though in only 5 of the 9 mice with both.

## disengaged-laps-uniform
> Disengaged laps are spread evenly through RE low; their probability in the first half of RE low is similar to the second half (P = 0.14) and their distribution does not differ from uniform (P = 0.7).

In [ ]:
for L in K.LICK_READINGS:
    f1, f2, posn = [], [], []
    for s in WT:
        b = RES[s]["beh"]
        ls = b["lick_stop"][L]
        dis = ~b["engaged_ur"][ls:]
        n = len(dis)
        if n < 2:
            continue
        h = n // 2
        f1.append(dis[:h].mean())
        f2.append(dis[h:].mean())
        idx = np.where(dis)[0]
        posn += list((idx + 0.5) / n)
    f1, f2 = np.array(f1), np.array(f2)
    sw = stats.shapiro(f1 - f2).pvalue
    t = stats.ttest_rel(f1, f2)
    w = stats.wilcoxon(f1, f2)
    ks = stats.kstest(posn, "uniform")
    print(f"{L}: fraction disengaged first half {f1.mean():.3f}, second half {f2.mean():.3f} (n = {len(f1)} mice); Shapiro-Wilk p={sw:.3f}; "
          f"paired t p={t.pvalue:.3f}; Wilcoxon p={w.pvalue:.3f}; KS against uniform on {len(posn)} laps D={ks.statistic:.3f} p={ks.pvalue:.3f}")

**Result: reproduced.** The fraction of disengaged laps is 0.294 in the first and 0.298 in the second half of RE low (paired t p = 0.960; Wilcoxon p = 0.820), and their positions do not differ from uniform (KS p = 0.294), under L1; L2 and L3 give p from 0.26 to 1.

## Place fields: the mice of Figures 2 and 3

Figures 2 and 3 use the R-UR-RR mice with more than 12 engaged RE low laps (Methods, Engagement with VR, approach behavior). Place fields are defined separately in R (all laps), in the engaged RE low laps and in RR (all laps), by the criteria in the helper's docstring.

In [ ]:
def eng_laps(s, L, ekey="engaged_ur"):
    b = RES[s]["beh"]
    ls = b["lick_stop"][L]
    return [i for i in range(ls, len(b[ekey])) if b[ekey][i]]

FIG23 = {}
for L in K.LICK_READINGS:
    counts = {s: len(eng_laps(s, L)) for s in K.WT_FULL}
    FIG23[L] = [s for s in K.WT_FULL if counts[s] > 12]
    print(f"{L}: engaged RE low laps per R-UR-RR mouse {counts}; mice with more than 12: {FIG23[L]}")
print("place fields per mouse and lap set:")
print(pd.DataFrame({s: {k: (len(v["fields"]) if v else None) for k, v in RES[s]["fields"].items()} for s in WT}).to_string())

def fields_df(s, name):
    f = RES[s]["fields"].get(name)
    if not f:
        return pd.DataFrame(columns=["cell", "start", "end", "reliability", "out_in", "width", "intensity", "com", "frac_laps"])
    d = pd.DataFrame(f["fields"])
    d["mouse"] = s
    return d

def cell_com(df):
    """COM of each cell's strongest field."""
    return df.sort_values("intensity").groupby("cell")["com"].last()

def mean_map(s, cond, cells, laps=None):
    m = RES[s]["maps"][cond][cells]
    if laps is not None:
        m = m[:, laps]
    return m.mean(axis=1)

def rowcorr(a, b):
    a = a - a.mean(1, keepdims=True)
    b = b - b.mean(1, keepdims=True)
    den = np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1))
    with np.errstate(invalid="ignore", divide="ignore"):
        return (a * b).sum(1) / den

def group_of(com):
    return np.where(com < 50, "start", np.where(com < 150, "middle", "end"))

## fig2c-spatial-correlation-drop
> Place fields defined in R partially remap in RE low (engaged laps), with a lower spatial correlation between R and RE low than within R, and remap less in RR.

Reading S1 (within-R from control mice) needs the control mice, which the dandiset does not hold; reading S2 (first against second half of the R laps of the same mice) is run.

In [ ]:
def spatial_corr(L, mice, ekey="engaged_ur", lowset="eng"):
    rows = []
    for s in mice:
        dfR = fields_df(s, "R")
        cells = np.array(sorted(dfR["cell"].unique()))
        nR = RES[s]["maps"]["Rewarded"].shape[1]
        mR = mean_map(s, "Rewarded", cells)
        h1 = mean_map(s, "Rewarded", cells, list(range(nR // 2)))
        h2 = mean_map(s, "Rewarded", cells, list(range(nR // 2, nR)))
        b = RES[s]["beh"]
        low = eng_laps(s, L, ekey) if lowset == "eng" else list(range(b["lick_stop"][L], len(b["engaged_ur"])))
        mlow = mean_map(s, "Unrewarded", cells, low)
        d = pd.DataFrame({"mouse": s, "cell": cells, "com": cell_com(dfR).reindex(cells).values,
                          "within R": rowcorr(h1, h2), "R vs RE low": rowcorr(mR, mlow)})
        if "ReRewarded" in RES[s]["maps"]:
            d["R vs RR"] = rowcorr(mR, mean_map(s, "ReRewarded", cells))
        rows.append(d)
    return pd.concat(rows, ignore_index=True)

fig2c = {}
for L in K.LICK_READINGS:
    sc = spatial_corr(L, FIG23[L])
    fig2c[L] = sc
    ok = sc[["within R", "R vs RE low", "R vs RR"]].dropna()
    t1 = stats.ttest_rel(ok["R vs RE low"], ok["within R"])
    t2 = stats.ttest_rel(ok["R vs RR"], ok["R vs RE low"])
    t3 = stats.ttest_rel(ok["R vs RR"], ok["within R"])
    print(f"--- {L}, mice {FIG23[L]}: {len(sc)} R place cells, {len(ok)} with all three correlations defined ({len(sc) - len(ok)} dropped: no activity in a lap set)")
    print(f"    mean within R {ok['within R'].mean():.3f}, R vs RE low {ok['R vs RE low'].mean():.3f}, R vs RR {ok['R vs RR'].mean():.3f}")
    print(f"    RE low vs within R t={t1.statistic:.2f} p={t1.pvalue:.2e}; RR vs RE low t={t2.statistic:.2f} p={t2.pvalue:.2e}; RR vs within R t={t3.statistic:.2f} p={t3.pvalue:.2e}")
    print("    per mouse means:\n", ok.join(sc["mouse"]).groupby("mouse").mean(numeric_only=True).round(3).to_string())

**Result: reproduced.** For the 605 R place cells of the 3 mice with correlations in every lap set, the correlation of R with RE low (0.566) is below the within-R correlation (0.872; paired t p = 6.9e-64), and R with RR (0.708) lies between them, above RE low (p = 2.4e-17); the same holds under L2 and L3. Only reading S2 (halves of the same R session) could be run: the control mice of reading S1 are not in the dandiset.

## fig2d-remapping-all-locations
> The drop in spatial correlation in RE low occurs at all track locations, not only near the reward site.

In [ ]:
fig2d = {}
for L in K.LICK_READINGS:
    sc = fig2c[L].dropna(subset=["within R", "R vs RE low"])
    res_g = {}
    for g in ["start", "middle", "end"]:
        d = sc[group_of(sc["com"].values) == g]
        t = stats.ttest_rel(d["R vs RE low"], d["within R"])
        res_g[g] = (len(d), d["within R"].mean(), d["R vs RE low"].mean(), t.statistic, t.pvalue)
        print(f"{L} {g:6s}: n={len(d)} cells, within R {d['within R'].mean():.3f}, R vs RE low {d['R vs RE low'].mean():.3f}, paired t={t.statistic:.2f} p={t.pvalue:.2e}")
    fig2d[L] = res_g
    print(f"{L}: groups with RE low lower at p < 0.01: {sum(v[2] < v[1] and v[4] < 0.01 for v in res_g.values())} of 3")

**Result: reproduced.** The R against RE low correlation is below within-R at the start (0.522 against 0.884, p = 1.8e-16), middle (0.625 against 0.896, p = 6.2e-27) and end (0.515 against 0.834, p = 2.6e-24) under L1, 3 of 3 groups under every lick stop reading (reading S2 only).

## fig3a-field-numbers-similar and the parameters of Figure 3b
> The total number of place fields tiling the track is similar in R, RE low (engaged laps) and RR (605, 663 and 674 fields in 3 mice).

In [ ]:
PF = {}
for L in K.LICK_READINGS:
    PF[L] = {"R": pd.concat([fields_df(s, "R") for s in FIG23[L]], ignore_index=True),
             "RE low": pd.concat([fields_df(s, f"RElow_eng_{L}") for s in FIG23[L]], ignore_index=True),
             "RR": pd.concat([fields_df(s, "RR") for s in FIG23[L]], ignore_index=True)}
    n = {k: len(v) for k, v in PF[L].items()}
    within = {k: abs(n[k] - v) <= tol for k, (v, tol) in {"R": (605, 60), "RE low": (663, 66), "RR": (674, 67)}.items()}
    print(f"{L}, mice {FIG23[L]}: fields R {n['R']}, RE low engaged {n['RE low']}, RR {n['RR']}; within tolerance {within}")
    print("    per mouse:", {k: v.groupby("mouse").size().to_dict() for k, v in PF[L].items()})
    print("    place cells (cells with a field) per condition:", {k: v.groupby("mouse")["cell"].nunique().sum() for k, v in PF[L].items()})

**Result: not reproduced.** In the 3 mice, R has 608 fields and RR 631, within tolerance of 605 and 674, but the engaged RE low laps give 265 (L1) to 257 (L3) fields against 663, so the numbers are not similar.

## fig3b-reliability, fig3b-out-in-ratio, fig3b-width-intensity
> Place field trial to trial reliability falls from 0.47 in R to 0.32 in RE low (engaged laps) and returns towards R in RR (0.41). The out/in place field firing ratio rises from 0.10 in R to 0.16 in RE low and returns towards R in RR (0.12). Place fields widen from 53.34 cm in R to 60.56 cm in RE low (54.08 in RR), and firing intensity falls slightly from 0.36 to 0.31 and stays low in RR (0.29).

In [ ]:
EXP3B = {"reliability": {"R": (0.47, 0.10), "RE low": (0.32, 0.17), "RR": (0.41, 0.10)},
         "out_in": {"R": (0.10, 0.03), "RE low": (0.16, 0.04), "RR": (0.12, 0.04)},
         "width": {"R": (53.34, 9.47), "RE low": (60.56, 12.5), "RR": (54.08, 7.23)},
         "intensity": {"R": (0.36, 0.05), "RE low": (0.31, 0.09), "RR": (0.29, 0.05)}}
fig3b = {}
for L in K.LICK_READINGS:
    print(f"--- {L}")
    for p, ex in EXP3B.items():
        for reading in ["per-mouse means", "pooled fields"]:
            vals = {}
            for c in ["R", "RE low", "RR"]:
                d = PF[L][c]
                if reading == "per-mouse means":
                    pm = d.groupby("mouse")[p].mean()
                    vals[c] = mean_ci(pm)
                else:
                    vals[c] = mean_ci(d[p])
            within = {c: abs(vals[c][0] - ex[c][0]) <= ex[c][1] for c in ex}
            fig3b[(L, p, reading)] = (vals, within)
            print(f"  {p:11s} {reading:16s}: " + "; ".join(f"{c} {m:.3f} +/- {h:.3f}" for c, (m, h) in vals.items()) + f"  within tolerance {within}")
        a, b2 = PF[L]["R"][p].dropna(), PF[L]["RE low"][p].dropna()
        t = stats.ttest_ind(a, b2)
        ks = stats.ks_2samp(a, b2)
        print(f"  {p:11s} R vs RE low pooled fields: unpaired t={t.statistic:.2f} p={t.pvalue:.2e}; KS D={ks.statistic:.3f} p={ks.pvalue:.2e}; range R {a.min():.2f} to {a.max():.2f}, RE low {b2.min():.2f} to {b2.max():.2f}")
print("per mouse (L1):")
print(pd.concat({c: PF["L1"][c].groupby("mouse")[["reliability", "out_in", "width", "intensity"]].mean() for c in PF["L1"]}).round(3).to_string())

**Result: reproduced.** Over the per-mouse means, reliability is 0.556 in R, 0.321 in RE low and 0.493 in RR (L1), within the tolerances of 0.47, 0.32 and 0.41, and RE low is below R (pooled fields, unpaired t p = 5.0e-66; KS p = 7.3e-47).

**Result: reproduced.** Under L2, the per-mouse means of the out/in ratio are R 0.075, RE low 0.120 and RR 0.083, within the tolerances of 0.10, 0.16 and 0.12, and RE low is above R (pooled fields, unpaired t p = 1.4e-27). Under L1 and L3 RE low (0.119 and 0.117) falls just below its tolerance band.

**Result: not reproduced.** Fields are much wider than the paper's: per-mouse means of 93.8 cm in R, 97.4 cm in RE low and 84.6 cm in RR, against 53.34, 60.56 and 54.08 cm; the RE low widening is small (pooled p = 0.038). Firing intensity is within tolerance in R (0.395) and RE low (0.288) over per-mouse means, and falls from R to RE low (p = 4.5e-09), but RR (0.371) is above its tolerance of 0.29 +/- 0.05.

## fig3c-degradation-all-locations
> The degradation of place fields in RE low (reliability and out/in ratio) is seen at all track locations.

In [ ]:
for L in K.LICK_READINGS:
    n_ok = {}
    for p, direction in [("reliability", -1), ("out_in", 1)]:
        k = 0
        for g in ["start", "middle", "end"]:
            a = PF[L]["R"][group_of(PF[L]["R"]["com"].values) == g][p].dropna()
            b2 = PF[L]["RE low"][group_of(PF[L]["RE low"]["com"].values) == g][p].dropna()
            ks = stats.ks_2samp(a, b2)
            ok = (np.sign(b2.mean() - a.mean()) == direction) and ks.pvalue < 0.01
            k += ok
            print(f"{L} {p:11s} {g:6s}: R {a.mean():.3f} (n={len(a)}), RE low {b2.mean():.3f} (n={len(b2)}), KS D={ks.statistic:.3f} p={ks.pvalue:.2e}, holds {ok}")
        n_ok[p] = k
    print(f"{L}: groups that hold, reliability {n_ok['reliability']} of 3, out/in {n_ok['out_in']} of 3")

**Result: partially reproduced.** Under L1, reliability is lower in RE low at the start, middle and end (KS p 2.1e-12 to 1.7e-26), and the out/in ratio is higher in the middle (p = 9.8e-13) and end (p = 1.4e-10) but not at the start (p = 0.024 against the 0.01 threshold), 3 groups for one measure and 2 for the other; L2 and L3 give the same.

## supp11-all-mice-parameters
> With all mice and both engaged and disengaged RE low laps, place field medians show the same degradation and return.

In [ ]:
EXP11 = {"reliability": (0.48, 0.26, 0.44), "out_in": (0.10, 0.19, 0.13), "width": (50.21, 54.96, 45.71), "intensity": (0.34, 0.26, 0.28)}
supp11 = {}
for L in K.LICK_READINGS:
    pf = {"R": pd.concat([fields_df(s, "R") for s in WT]), "RE low": pd.concat([fields_df(s, f"RElow_all_{L}") for s in WT]),
          "RR": pd.concat([fields_df(s, "RR") for s in K.WT_FULL])}
    for reading in ["(a) median of per-mouse means", "(b) median of pooled fields"]:
        n_in = 0
        dirs = []
        line = []
        for p, ex in EXP11.items():
            v = []
            for c in ["R", "RE low", "RR"]:
                if reading.startswith("(a)"):
                    v.append(pf[c].groupby("mouse")[p].mean().median())
                else:
                    v.append(pf[c][p].median())
            n_in += sum(abs(x - e) <= 0.1 * e for x, e in zip(v, ex))
            dirs.append((v[1] < v[0]) if p in ("reliability", "intensity") else (v[1] > v[0]))
            line.append(f"{p} " + "/".join(f"{x:.3f}" for x in v))
        supp11[(L, reading)] = (n_in, all(dirs))
        print(f"{L} {reading}: " + "; ".join(line) + f" | within 10%: {n_in} of 12; directions hold: {all(dirs)} {dirs}")
    print(f"    fields: R {len(pf['R'])} (10 mice), RE low {len(pf['RE low'])}, RR {len(pf['RR'])} (5 mice)")

**Result: not reproduced.** With all 10 mice and all RE low laps, every direction holds (reliability and intensity lower and out/in ratio and width higher in RE low), but only 3 of the 12 medians are within 10% under reading (b) and none under reading (a); the criterion asks for 10 (7 for partial). Widths are 75 to 80 cm against 46 to 55 cm.

## fig2e-place-field-fates
> Of 605 R place fields (3 mice), 169 (27.9%) stay stable through R, RE low and RR; in RE low 222 (36.7%) stay stable, 249 (41.2%) disappear and 134 (22.1%) remap, and 296 of the 663 RE low fields (44.6%) are new. In RR 311 R fields (51.4%) are stable and 89 (14.7%) remapped, 126 of the 249 lost fields (50.6%) reappear, 64 of the 296 new RE low fields (21.6%) stay and 204 (68.9%) disappear, and 140 of 674 RR fields (20.8%) are new.

In [ ]:
def is_stable(f_early, f_late, rule):
    if rule == "F1":
        return f_late["start"] < f_early["end"] and f_early["start"] < f_late["end"]
    if rule == "F2":
        return f_early["start"] * K.BIN <= f_late["com"] < f_early["end"] * K.BIN
    return group_of(np.array([f_early["com"]]))[0] == group_of(np.array([f_late["com"]]))[0]

def match(f, later_cell_fields):
    if later_cell_fields is None or len(later_cell_fields) == 0:
        return None
    i = np.argmin(np.abs(later_cell_fields["com"].values - f["com"]))
    return later_cell_fields.iloc[i]

def fates(L, rule):
    out = {k: 0 for k in ["R", "RE low", "RR", "stable through", "stable RE low", "lost RE low", "remap RE low", "new RE low",
                          "stable RR", "remap RR", "lost RR", "lost reappear RR", "new low stable RR", "new low lost RR", "new RR (vs RE low)", "new RR (vs R and RE low)",
                          "lost end", "lost end reappear"]}
    newlow_bins = np.zeros(4, int)
    newrr_bins = np.zeros(4, int)
    for s in FIG23[L]:
        R, Lo, RR = fields_df(s, "R"), fields_df(s, f"RElow_eng_{L}"), fields_df(s, "RR")
        gR, gL, gRR = R.groupby("cell"), Lo.groupby("cell"), RR.groupby("cell")
        cR, cL, cRR = set(R["cell"]), set(Lo["cell"]), set(RR["cell"])
        out["R"] += len(R); out["RE low"] += len(Lo); out["RR"] += len(RR)
        for _, f in R.iterrows():
            ml = match(f, gL.get_group(f["cell"]) if f["cell"] in cL else None)
            mr = match(f, gRR.get_group(f["cell"]) if f["cell"] in cRR else None)
            st_low = ml is not None and is_stable(f, ml, rule)
            st_rr = mr is not None and is_stable(f, mr, rule)
            if ml is None:
                out["lost RE low"] += 1
                if f["com"] >= 150:
                    out["lost end"] += 1
                if st_rr:
                    out["lost reappear RR"] += 1
                    if f["com"] >= 150:
                        out["lost end reappear"] += 1
            elif st_low:
                out["stable RE low"] += 1
            else:
                out["remap RE low"] += 1
            if mr is None:
                out["lost RR"] += 1
            elif st_rr:
                out["stable RR"] += 1
            else:
                out["remap RR"] += 1
            if st_low and st_rr:
                out["stable through"] += 1
        for _, f in Lo.iterrows():
            if f["cell"] not in cR:
                out["new RE low"] += 1
                newlow_bins[min(int(f["com"] // 50), 3)] += 1
                mr = match(f, gRR.get_group(f["cell"]) if f["cell"] in cRR else None)
                if mr is None:
                    out["new low lost RR"] += 1
                elif is_stable(f, mr, rule):
                    out["new low stable RR"] += 1
        for _, f in RR.iterrows():
            if f["cell"] not in cL:
                out["new RR (vs RE low)"] += 1
                if f["cell"] not in cR:
                    out["new RR (vs R and RE low)"] += 1
                    newrr_bins[min(int(f["com"] // 50), 3)] += 1
    return out, newlow_bins, newrr_bins

FATES = {}
for L in K.LICK_READINGS:
    for rule in ["F1", "F2", "F3"]:
        o, nlb, nrb = fates(L, rule)
        FATES[(L, rule)] = (o, nlb, nrb)
        pct = lambda a, b: 100 * a / b if b else np.nan
        vals = {"stable through": pct(o["stable through"], o["R"]), "stable RE low": pct(o["stable RE low"], o["R"]),
                "lost RE low": pct(o["lost RE low"], o["R"]), "remap RE low": pct(o["remap RE low"], o["R"]),
                "new RE low": pct(o["new RE low"], o["RE low"]), "stable RR": pct(o["stable RR"], o["R"]),
                "remap RR": pct(o["remap RR"], o["R"]), "lost reappear": pct(o["lost reappear RR"], o["lost RE low"]),
                "new low stable RR": pct(o["new low stable RR"], o["new RE low"]), "new low lost RR": pct(o["new low lost RR"], o["new RE low"]),
                "new RR (vs R and RE low)": pct(o["new RR (vs R and RE low)"], o["RR"]), "new RR (vs RE low)": pct(o["new RR (vs RE low)"], o["RR"])}
        FATES[(L, rule)] = (o, nlb, nrb, vals)
        print(f"{L} {rule}: fields R {o['R']}, RE low {o['RE low']}, RR {o['RR']} | " + "; ".join(f"{k} {v:.1f}%" for k, v in vals.items()))
EXP2E = {"stable through": (27.9, 2.8), "stable RE low": (36.7, 3.7), "lost RE low": (41.2, 4.1), "remap RE low": (22.1, 2.2), "new RE low": (44.6, 4.5),
         "stable RR": (51.4, 5.1), "remap RR": (14.7, 1.5), "lost reappear": (50.6, 5.1), "new low stable RR": (21.6, 2.2), "new low lost RR": (68.9, 6.9),
         "new RR (vs R and RE low)": (20.8, 2.1)}
for k, (o, nlb, nrb, vals) in FATES.items():
    w = [n for n, (v, tol) in EXP2E.items() if abs(vals[n] - v) <= tol]
    print(k, f"within tolerance: {len(w)} of {len(EXP2E)}:", w)

**Result: not reproduced.** The R (608) and RR (631) field counts match, but RE low engaged laps give 265 fields (paper 663), 80.8% of the R fields disappear in RE low (paper 41.2%), and 55.5% of RE low fields are new (paper 44.6%); these do not depend on the stable/remapped definitions. Under F1 to F3, 0 to 2 of the 11 percentages are within tolerance under any lick stop reading.

## reward-zone-field-turnover
> Of the R place fields that disappear in RE low, 41.8% (104 of 249) are at the track end (150 to 200 cm); new RE low fields are spread evenly (26, 28, 21 and 25% per 50 cm). In RR only 61 of the 104 lost end fields return (58.6%), and new RR fields cluster at the end (31, 24, 18 and 67 of 140, 48% at 150 to 200 cm).

In [ ]:
for k, (o, nlb, nrb, vals) in FATES.items():
    print(f"{k}: lost {o['lost RE low']}, of them at 150 to 200 cm {o['lost end']} ({100 * o['lost end'] / max(1, o['lost RE low']):.1f}%); "
          f"new RE low per 50 cm {nlb.tolist()} ({np.round(100 * nlb / max(1, nlb.sum())).astype(int).tolist()}%); "
          f"lost end fields reappearing in RR {o['lost end reappear']}; new RR per 50 cm {nrb.tolist()} ({np.round(100 * nrb / max(1, nrb.sum())).astype(int).tolist()}%)")

**Result: not reproduced.** Of the R fields lost in RE low (491 under L1), 175 (35.6%) are at 150 to 200 cm, against 104 of 249 (41.8%); new RE low fields are spread evenly (24, 27, 22 and 27%, close to the paper's 26, 28, 21 and 25%) but number 147, not 296, so every count is outside tolerance; 68 to 75 lost end fields reappear in RR (paper 61), and new RR fields are 78, 37, 44 and 98 per 50 cm (paper 31, 24, 18 and 67).

## fig3d-reward-overrepresentation
> Place fields over-represent the reward site at the track end in R; the over-representation disappears in RE low and reappears in RR. Gaussian fits to field COMs are 114 +/- 55 cm in R, 102 +/- 54 in UR and 112 +/- 58 in RR.

In [ ]:
def end_minus_middle(df):
    com = df["com"].dropna().values
    if len(com) == 0:
        return np.nan
    h = np.histogram(com, bins=[0, 50, 100, 150, 200])[0] / len(com) * 100
    return h[3] - h[1:3].mean()

for L in K.LICK_READINGS:
    em = {c: [end_minus_middle(PF[L][c][PF[L][c]["mouse"] == s]) for s in FIG23[L]] for c in PF[L]}
    gauss = {c: stats.norm.fit(PF[L][c]["com"].dropna()) for c in PF[L]}
    print(f"--- {L}, mice {FIG23[L]}")
    print("   end minus middle (% of fields) per mouse:", {c: np.round(v, 1).tolist() for c, v in em.items()})
    print("   Gaussian fit mean +/- SD (cm):", {c: (round(m, 1), round(sd, 1)) for c, (m, sd) in gauss.items()})
    for c, v in em.items():
        t = stats.ttest_1samp(v, 0)
        print(f"   {c}: mean {np.mean(v):.1f}, t-test against 0 p={t.pvalue:.3f}")
    print("   R > 0 in all mice:", all(np.array(em["R"]) > 0), "| RR > 0 in all mice:", all(np.array(em["RR"]) > 0),
          "| RE low below R in all mice:", all(np.array(em["RE low"]) < np.array(em["R"])), "| RE low mean <= 0:", np.mean(em["RE low"]) <= 0)

**Result: partially reproduced.** The Gaussian fits are within tolerance (R 113.4 +/- 59.4, RE low 106.7 +/- 54.3, RR 111.2 +/- 60.2 cm under L1). The end minus middle excess is above 0 in R in 2 of the 3 mice (13.1, -4.2, 17.7) and in RR in all 3; in RE low its mean (2.7) is lower than in R (8.8) but above 0, and it is not lower than R in every mouse, under every lick stop reading.

## fig2a-pv-cluster-transition
> Lap by lap population vector correlations of R place cells show a transition around the lick stop (RE high to RE low) and again after reward returns; the probability of laps belonging to the R cluster dips from RE high to RE low, not from R to RE high, and rises to some extent in RR (fully only in mouse 1).

Per lap, the population vector is each R place cell's mean dF/F in each of the 40 bins. Laps: R, RE high, RE low engaged, RR. k-means with 3 clusters, run 1000 times from random starts (seeded); the R cluster is the one holding most R laps in each run.

In [ ]:
from sklearn.cluster import KMeans
rng_pv = np.random.default_rng(2)
fig2a = {}
for L in K.LICK_READINGS:
    holds = 0
    for s in FIG23[L]:
        cells = np.array(sorted(fields_df(s, "R")["cell"].unique()))
        b = RES[s]["beh"]
        ls = b["lick_stop"][L]
        parts = [("R", "Rewarded", list(range(RES[s]["maps"]["Rewarded"].shape[1]))),
                 ("RE high", "Unrewarded", list(range(ls))),
                 ("RE low", "Unrewarded", eng_laps(s, L)),
                 ("RR", "ReRewarded", list(range(RES[s]["maps"]["ReRewarded"].shape[1])))]
        X, lab = [], []
        for name, cond, laps in parts:
            m = RES[s]["maps"][cond][cells][:, laps]  # cells x laps x 40
            X.append(m.transpose(1, 0, 2).reshape(len(laps), -1))
            lab += [name] * len(laps)
        X = np.vstack(X)
        lab = np.array(lab)
        pvc = np.corrcoef(X)
        prob = np.zeros(len(X))
        n_runs = 1000
        for i in range(n_runs):
            km = KMeans(3, n_init=1, random_state=int(rng_pv.integers(1 << 31))).fit(X)
            rc = np.bincount(km.labels_[lab == "R"], minlength=3).argmax()
            prob += km.labels_ == rc
        prob /= n_runs
        mp = {n: prob[lab == n].mean() for n in ["R", "RE high", "RE low", "RR"]}
        ok = abs(mp["RE high"] - mp["R"]) <= 0.2 and mp["RE low"] <= mp["RE high"] - 0.3
        holds += ok
        mean_pvc = {n: pvc[np.ix_(lab == n, lab == "R")].mean() for n in ["R", "RE high", "RE low", "RR"]}
        print(f"{L} {s}: {len(cells)} R place cells, laps {dict(zip(*np.unique(lab, return_counts=True)))}; mean R-cluster probability "
              + ", ".join(f"{n} {v:.2f}" for n, v in mp.items()) + f"; holds {ok}")
        print("    mean PV correlation with R laps: " + ", ".join(f"{n} {v:.3f}" for n, v in mean_pvc.items()))
        print("    lap-wise probability:", " ".join(f"{x:.2f}" for x in prob))
    fig2a[L] = holds
    print(f"{L}: holds in {holds} of {len(FIG23[L])} mice")

**Result: reproduced.** Under L1 and L2 the R-cluster probability stays near R in RE high and drops by more than 0.3 in RE low in 2 of the 3 mice (NR21: R 0.84, RE high 0.80, RE low 0.23; CFC4: 0.92, 0.77, 0.12); in NR23, with 2 RE high laps, RE high is already low (0.52). Under L3 the rule holds in 1 mouse. RR probabilities are low (0.02, 0.26, 0.31), so the partial return in RR is weak here.

## supp13-novel-environment
> In a novel, never reinforced environment, decoding and place field reliability and out/in ratio are poor in early laps and improve over the session as mice develop reward expectation, becoming comparable to R and RR; RE low reliability and out/in ratio match the early novel laps.

Five of the six mice with a novel session are in the dandiset. Early = first 12 novel laps, late = last 12. Decoders are trained and tested within each condition (sliding 20% test block), each condition down-sampled to the fewest laps; for early and late R2 the novel session is decoded on all its laps. Place field parameters in a block are read two ways: fields defined in the block, or fields defined over the whole novel session and measured on the block's laps.

In [ ]:
def block_params(s, name_all, laps):
    """Parameters of the session's novel fields measured on a block of laps."""
    df = fields_df(s, name_all)
    M = RES[s]["maps"]["Novel"][:, laps]
    out = []
    for _, f in df.iterrows():
        lm = M[int(f["cell"])]
        p = K.field_parameters(lm, lm > 0, int(f["start"]), int(f["end"]))
        out.append(p)
    return pd.DataFrame(out)

def prelicks_per_lap(c, zone):
    """Licks in the reward zone before the reward is delivered, per lap (pre-licking)."""
    out = []
    for a, e in K.complete_laps(c):
        rw = c["rewards"][(c["rewards"] >= a) & (c["rewards"] < e)]
        stop = rw[0] if len(rw) else e
        lk = c["licks"][(c["licks"] >= a) & (c["licks"] < stop)]
        out.append(int((c["pos"][lk] >= zone).sum()))
    return np.array(out)

rows = []
for s in K.WT_FULL:
    b = RES[s]["beh"]
    cN = b["conds"]["Novel"]
    nN = len(K.complete_laps(cN))
    early, late = list(range(min(12, nN))), list(range(max(0, nN - 12), nN))
    zl = prelicks_per_lap(cN, b["zone"])
    cv = RES[s]["dec_cv_novel_all"]
    r = {"mouse": s, "novel laps": nN,
         "pre-licking early": np.mean(zl[early]), "pre-licking late": np.mean(zl[late]),
         "R2 early": np.nanmean(cv[early]), "R2 late": np.nanmean(cv[late]),
         "R2 R (cv)": np.nanmean(RES[s]["dec_cv"]["Rewarded"]), "R2 RR (cv)": np.nanmean(RES[s]["dec_cv"]["ReRewarded"]),
         "R2 N (cv)": np.nanmean(RES[s]["dec_cv"]["Novel"])}
    for p in ["reliability", "out_in"]:
        r[f"{p} early (block fields)"] = fields_df(s, "N_early")[p].mean()
        r[f"{p} late (block fields)"] = fields_df(s, "N_late")[p].mean()
        r[f"{p} early (session fields)"] = block_params(s, "N", early)[p].mean()
        r[f"{p} late (session fields)"] = block_params(s, "N", late)[p].mean()
        r[f"{p} R"] = fields_df(s, "R")[p].mean()
        r[f"{p} RR"] = fields_df(s, "RR")[p].mean()
        r[f"{p} RE low (L1, all laps)"] = fields_df(s, "RElow_all_L1")[p].mean()
    rows.append(r)
nov = pd.DataFrame(rows).set_index("mouse")
print(nov.T.round(3).to_string())
tests = {}
for label, a, b2, better in [("pre-licking", "pre-licking early", "pre-licking late", 1), ("decoder R2", "R2 early", "R2 late", 1),
                             ("reliability (block fields)", "reliability early (block fields)", "reliability late (block fields)", 1),
                             ("reliability (session fields)", "reliability early (session fields)", "reliability late (session fields)", 1),
                             ("out/in (block fields)", "out_in early (block fields)", "out_in late (block fields)", -1),
                             ("out/in (session fields)", "out_in early (session fields)", "out_in late (session fields)", -1)]:
    d = nov[[a, b2]].dropna()
    t = stats.ttest_rel(d[b2], d[a])
    ok = np.sign(d[b2].mean() - d[a].mean()) == better and t.pvalue < 0.05
    tests[label] = ok
    print(f"{label:30s}: early {d[a].mean():.3f}, late {d[b2].mean():.3f}, paired t={t.statistic:.2f} p={t.pvalue:.4f} (n={len(d)}); improves at p < 0.05: {ok}")

**Result: partially reproduced.** In the 5 novel mice, late laps decode better than early laps (R2 0.821 to 0.919, p = 0.0093) and have higher reliability (block fields 0.304 to 0.391, p = 0.016; session fields 0.229 to 0.425, p = 0.0036). Pre-licking (p = 0.38) and the out/in ratio (p = 0.20 and 0.11) do not change significantly: 2 of the 4 tests hold.

## DREADD mice

Four DAT-Cre hM4D(Gi) mice (sub-009, 011, 012, 014); the paper has six (five for saline). Day 1 (R, UR, RR) and days 2 to 4 (before and after saline, CNO and DCZ) are stored back to back in one file per mouse; 012 has no RR.

In [ ]:
PAIRS = {"saline": ("BeforeSaline", "AfterSaline"), "DCZ": ("BeforeDCZ", "AfterDCZ"), "CNO": ("BeforeCNO", "AfterCNO")}
for s in DR:
    b = RES[s]["beh"]
    print(s, {n: len(K.complete_laps(c)) for n, c in b["conds"].items()}, "lick stop", b["lick_stop"], "engaged RE low (L1)", len(eng_laps(s, "L1")))

## supp16-dreadd-running-speed
> DCZ and CNO slow lap running speed in R (from about 42 to about 26), while saline does not, and approach behaviour and anticipatory licking remain.

In [ ]:
EXP16 = {"BeforeSaline": (42.74, 1.62), "AfterSaline": (42.75, 1.42), "BeforeDCZ": (41.99, 1.74), "AfterDCZ": (25.96, 1.66), "BeforeCNO": (43.18, 1.92), "AfterCNO": (25.71, 1.68)}

def run_speed(c):
    return np.array([np.mean(c["speed"][a:e][c["speed"][a:e] >= K.RUN_SPEED]) for a, e in K.complete_laps(c)])

def prelick_laps(c, zone):
    """Fraction of laps with a lick in the reward zone before the reward."""
    out = []
    for a, e in K.complete_laps(c):
        rw = c["rewards"][(c["rewards"] >= a) & (c["rewards"] < e)]
        stop = rw[0] if len(rw) else e
        lk = c["licks"][(c["licks"] >= a) & (c["licks"] < stop)]
        out.append(bool((c["pos"][lk] >= zone).any()))
    return np.mean(out)

for measure, fn in [("lap speed 200 cm / lap time", lambda c: K.lap_speeds(c)), ("mean running speed", run_speed)]:
    print(f"--- {measure} (cm/s)")
    for reading in ["laps pooled", "per-mouse means"]:
        vals = {}
        for n in EXP16:
            per = [fn(RES[s]["beh"]["conds"][n]) for s in DR]
            vals[n] = mean_ci(np.concatenate(per)) if reading == "laps pooled" else mean_ci([np.nanmean(p) for p in per])
        within = {n: abs(vals[n][0] - v) <= tol for n, (v, tol) in EXP16.items()}
        print(f"  {reading}: " + "; ".join(f"{n} {m:.2f} [{m - h:.2f} {m + h:.2f}]" for n, (m, h) in vals.items()) + f" | within {sum(within.values())} of 6")
    for drug, (bn, an) in PAIRS.items():
        before = np.concatenate([fn(RES[s]["beh"]["conds"][bn]) for s in DR])
        after = np.concatenate([fn(RES[s]["beh"]["conds"][an]) for s in DR])
        t = stats.ttest_ind(after[np.isfinite(after)], before[np.isfinite(before)])
        print(f"  {drug}: after minus before {np.nanmean(after) - np.nanmean(before):.2f}, unpaired t over laps t={t.statistic:.2f} p={t.pvalue:.2e} ({np.isfinite(before).sum()} and {np.isfinite(after).sum()} laps)")
for s in DR:
    print(s, "lap speed per condition:", {n: round(np.nanmean(K.lap_speeds(RES[s]["beh"]["conds"][n])), 1) for n in EXP16},
          "| laps with a pre-reward lick in the zone:", {n: round(prelick_laps(RES[s]["beh"]["conds"][n], RES[s]["beh"]["zone"]), 2) for n in EXP16},
          "| approach ratio:", {n: round(np.nanmean(RES[s]["beh"]["ratios"][n]), 2) for n in EXP16})

**Result: partially reproduced.** DCZ and CNO lower lap speed (after minus before -13.70 and -13.05 cm/s, unpaired t p = 7.9e-38 and 2.0e-29) and saline does not (+0.22, p = 0.78), and nearly all laps after DCZ and CNO still have a pre-reward lick in the zone. The values are outside tolerance: 25.9 to 27.6 cm/s before and 13.5 to 14.6 cm/s after, against about 42 and 26, in 4 of the 6 mice.

## fig4f-dreadd-spatial-correlation
> Spatial correlation of place fields before and after DCZ (and CNO) falls at all track locations, as in RE low, while it does not fall after saline.

In [ ]:
def pair_corr(s, first, second, second_laps=None):
    df = fields_df(s, first)
    cells = np.array(sorted(df["cell"].unique()))
    c1 = first if first in RES[s]["maps"] else RES[s]["fields"][first]["cond"]
    laps1 = None if first in RES[s]["maps"] else RES[s]["fields"][first]["laps"]
    m1 = mean_map(s, c1, cells, laps1)
    m2 = mean_map(s, second, cells, second_laps)
    return pd.DataFrame({"mouse": s, "cell": cells, "com": cell_com(df).reindex(cells).values, "r": rowcorr(m1, m2)})

def dreadd_corrs(L, lowset):
    out = {}
    rows = []
    for s in DR:
        b = RES[s]["beh"]
        low = eng_laps(s, L) if lowset == "engaged" else list(range(b["lick_stop"][L], len(b["engaged_ur"])))
        rows.append(pair_corr(s, "Rewarded", "Unrewarded", low))
    out["R/RE low"] = pd.concat(rows, ignore_index=True)
    for drug, (bn, an) in PAIRS.items():
        out[drug] = pd.concat([pair_corr(s, bn, an) for s in DR], ignore_index=True)
    return out

EXP4F = {"R/RE low": (928, 93), "saline": (1139, 114), "DCZ": (1629, 163), "CNO": (1498, 150)}
fig4f = {}
for L in K.LICK_READINGS:
    for lowset in ["engaged", "all"]:
        co = dreadd_corrs(L, lowset)
        fig4f[(L, lowset)] = co
        print(f"--- {L}, RE low {lowset} laps")
        print("   place cells:", {k: (len(v), "within" if abs(len(v) - EXP4F[k][0]) <= EXP4F[k][1] else "outside") for k, v in co.items()},
              "| mean r:", {k: round(v["r"].mean(), 3) for k, v in co.items()})
        for drug in ["DCZ", "CNO"]:
            ks = stats.ks_2samp(co[drug]["r"].dropna(), co["saline"]["r"].dropna())
            print(f"   {drug} vs saline overall: mean {co[drug]['r'].mean():.3f} vs {co['saline']['r'].mean():.3f}, KS D={ks.statistic:.3f} p={ks.pvalue:.2e}")
        for g in ["start", "middle", "end"]:
            sel = {k: v[group_of(v["com"].values) == g]["r"].dropna() for k, v in co.items()}
            line = []
            for k in ["saline", "DCZ", "CNO"]:
                ks = stats.ks_2samp(sel["R/RE low"], sel[k])
                line.append(f"R/RE low vs {k} KS p={ks.pvalue:.1e}")
            ks2 = {k: stats.ks_2samp(sel[k], sel["saline"]).pvalue for k in ["DCZ", "CNO"]}
            print(f"   {g:6s}: mean r " + ", ".join(f"{k} {v.mean():.3f} (n={len(v)})" for k, v in sel.items()) + " | " + "; ".join(line)
                  + f"; DCZ vs saline p={ks2['DCZ']:.1e}, CNO vs saline p={ks2['CNO']:.1e}")
print("per mouse mean r (L1, engaged):")
print(pd.DataFrame({k: v.groupby("mouse")["r"].mean() for k, v in fig4f[("L1", "engaged")].items()}).round(3).to_string())

**Result: partially reproduced.** Spatial correlation falls more after DCZ (0.657) and CNO (0.653) than after saline (0.793; KS p = 1.9e-16 and 3.3e-12), and at every position saline stays above R/RE low and DCZ and CNO lie below saline, under every reading. The place cell counts (610, 676, 842 and 641) are outside the tolerances of 928, 1139, 1629 and 1498; the dandiset holds 4 of the 6 mice.

## supp18-saline-slow-laps
> Splitting saline data into fast (before) and slow (after) laps gives only a small drop in spatial correlation, much smaller than DCZ or CNO, and no change in reliability (0.51 against 0.45) or out/in ratio (0.09 against 0.09).

Percentiles are taken within each mouse; the pooled-percentile reading was not run.

In [ ]:
for slow in ["Sal_slow_a", "Sal_slow_b"]:
    sp_f = np.concatenate([K.lap_speeds(RES[s]["beh"]["conds"]["BeforeSaline"])[RES[s]["fields"]["Sal_fast"]["laps"]] for s in DR])
    sp_s = np.concatenate([K.lap_speeds(RES[s]["beh"]["conds"]["AfterSaline"])[RES[s]["fields"][slow]["laps"]] for s in DR])
    pf_f = pd.concat([fields_df(s, "Sal_fast") for s in DR])
    pf_s = pd.concat([fields_df(s, slow) for s in DR])
    print(f"--- fast before saline against {slow} ({'below' if slow.endswith('a') else 'above'} the 40th percentile)")
    print(f"   lap speed fast {np.nanmean(sp_f):.2f} (paper 47.30 +/- 2.61), after {np.nanmean(sp_s):.2f} (paper 35.03 +/- 1.57) cm/s")
    for p, ex in [("reliability", (0.51, 0.06, 0.45, 0.13)), ("out_in", (0.09, 0.05, 0.09, 0.05))]:
        for reading in ["per-mouse means", "pooled fields"]:
            if reading == "per-mouse means":
                a, b2 = mean_ci(pf_f.groupby("mouse")[p].mean()), mean_ci(pf_s.groupby("mouse")[p].mean())
            else:
                a, b2 = mean_ci(pf_f[p]), mean_ci(pf_s[p])
            print(f"   {p} {reading}: fast {a[0]:.3f} +/- {a[1]:.3f}, after {b2[0]:.3f} +/- {b2[1]:.3f}; within tolerance {abs(a[0] - ex[0]) <= ex[1] and abs(b2[0] - ex[2]) <= ex[3]}")
    corr_split = pd.concat([pair_corr(s, "Sal_fast", "AfterSaline", RES[s]["fields"][slow]["laps"]) for s in DR])["r"].mean()
    co = fig4f[("L1", "engaged")]
    drop_split = co["saline"]["r"].mean() - corr_split
    drop_dcz = co["saline"]["r"].mean() - co["DCZ"]["r"].mean()
    drop_cno = co["saline"]["r"].mean() - co["CNO"]["r"].mean()
    print(f"   spatial correlation: saline pair {co['saline']['r'].mean():.3f}, fast vs slow split {corr_split:.3f}, DCZ pair {co['DCZ']['r'].mean():.3f}, CNO pair {co['CNO']['r'].mean():.3f}")
    print(f"   drop of the split {drop_split:.3f}; drop for DCZ {drop_dcz:.3f}, CNO {drop_cno:.3f}; split drop below half the DCZ drop: {drop_split < 0.5 * drop_dcz}")
    print("   fields: fast", len(pf_f), "after", len(pf_s), "| laps per mouse fast", [len(RES[s]["fields"]["Sal_fast"]["laps"]) for s in DR], "after", [len(RES[s]["fields"][slow]["laps"]) for s in DR])

**Result: not reproduced.** Splitting the saline days by lap speed gives fast laps at 30.63 cm/s (paper 47.30) and slow laps at 21.17 cm/s (reading a; paper 35.03); reliability is outside tolerance under both readings. The drop in spatial correlation for the split (0.207 under reading a, 0.136 under b) is not smaller than half the DCZ drop (0.136).

## fig4h-dreadd-reliability-outin
> DCZ and CNO lower place field reliability and raise out/in field firing, as RE low does, while saline does not.

In [ ]:
EXP4H = {"reliability": {"Rewarded": (0.41, 0.11), "RE low": (0.29, 0.12), "BeforeDCZ": (0.48, 0.10), "AfterDCZ": (0.41, 0.10), "BeforeCNO": (0.42, 0.11),
                         "AfterCNO": (0.35, 0.12), "BeforeSaline": (0.45, 0.11), "AfterSaline": (0.46, 0.08)},
         "out_in": {"Rewarded": (0.10, 0.04), "RE low": (0.17, 0.12), "BeforeDCZ": (0.10, 0.05), "AfterDCZ": (0.12, 0.04), "BeforeCNO": (0.10, 0.05),
                    "AfterCNO": (0.12, 0.05), "BeforeSaline": (0.09, 0.05), "AfterSaline": (0.09, 0.05)}}

def cdf_bins_test(a, b2, nb=20):
    edges = np.linspace(min(a.min(), b2.min()), max(a.max(), b2.max()), nb + 1)
    ca = np.cumsum(np.histogram(a, edges)[0]) / len(a)
    cb = np.cumsum(np.histogram(b2, edges)[0]) / len(b2)
    return stats.ttest_rel(ca, cb).pvalue

for L in K.LICK_READINGS:
    for lowset in ["engaged", "all"]:
        name = f"RElow_eng_{L}" if lowset == "engaged" else f"RElow_all_{L}"
        pf = {n: pd.concat([fields_df(s, n) for s in DR]) for n in ["Rewarded", "BeforeDCZ", "AfterDCZ", "BeforeCNO", "AfterCNO", "BeforeSaline", "AfterSaline"]}
        pf["RE low"] = pd.concat([fields_df(s, name) for s in DR])
        for reading in ["per-mouse means", "pooled fields"]:
            n_in = 0
            vals = {}
            for p, ex in EXP4H.items():
                for n, (v, tol) in ex.items():
                    m = mean_ci(pf[n].groupby("mouse")[p].mean())[0] if reading == "per-mouse means" else pf[n][p].mean()
                    vals[(p, n)] = m
                    n_in += abs(m - v) <= tol
            dirs = {d: (vals[("reliability", "After" + d)] < vals[("reliability", "Before" + d)], vals[("out_in", "After" + d)] > vals[("out_in", "Before" + d)]) for d in ["DCZ", "CNO"]}
            print(f"{L} RE low {lowset}, {reading}: within tolerance {n_in} of 16; DCZ/CNO (reliability lower, out/in higher) after: {dirs}")
            print("    " + "; ".join(f"{p[:3]} {n} {m:.3f}" for (p, n), m in vals.items()))
for p in ["reliability", "out_in"]:
    for d in ["Saline", "DCZ", "CNO"]:
        bm = [fields_df(s, "Before" + d)[p].mean() for s in DR]
        am = [fields_df(s, "After" + d)[p].mean() for s in DR]
        t = stats.ttest_rel(bm, am)
        a = pd.concat([fields_df(s, "Before" + d) for s in DR])[p].dropna().values
        b2 = pd.concat([fields_df(s, "After" + d) for s in DR])[p].dropna().values
        print(f"{p} {d}: per-mouse before {np.round(bm, 3).tolist()} after {np.round(am, 3).tolist()}; paired t over 4 mice p={t.pvalue:.3f}; paired t over cumulative histogram bins p={cdf_bins_test(a, b2):.3g}")

**Result: partially reproduced.** All 16 means are within tolerance under both readings. Out/in firing rises after DCZ and CNO, and reliability falls after DCZ (0.452 to 0.438 per mouse) but not after CNO (0.414 to 0.417) over per-mouse means; over pooled fields both fall. Saline is not without change: the out/in ratio falls after saline (paired t over 4 mice p = 0.013) and reliability differs over the cumulative histogram bins (p = 4.5e-06), so the two readings of the absence test disagree.

## fig4m-dreadd-overrepresentation
> DCZ and CNO tend to reduce place field over-representation of the reward site compared with saline, but not significantly (P = 0.06), and less than RE low does.

In [ ]:
EXP4M = {"BeforeSaline": 113, "AfterSaline": 114, "BeforeDCZ": 113, "AfterDCZ": 106, "BeforeCNO": 113, "AfterCNO": 109, "Rewarded": 110, "RE low": 103}
for L in K.LICK_READINGS:
    for lowset in ["engaged", "all"]:
        name = f"RElow_eng_{L}" if lowset == "engaged" else f"RElow_all_{L}"
        em = {}
        for s in DR:
            row = {n: end_minus_middle(fields_df(s, n)) for n in ["BeforeSaline", "AfterSaline", "BeforeDCZ", "AfterDCZ", "BeforeCNO", "AfterCNO", "Rewarded"]}
            row["RE low"] = end_minus_middle(fields_df(s, name))
            em[s] = row
        em = pd.DataFrame(em).T
        ch = pd.DataFrame({"saline": em["AfterSaline"] - em["BeforeSaline"], "DCZ": em["AfterDCZ"] - em["BeforeDCZ"],
                           "CNO": em["AfterCNO"] - em["BeforeCNO"], "RE low": em["RE low"] - em["Rewarded"]})
        t = stats.ttest_rel(ch["DCZ"], ch["saline"])
        w = stats.wilcoxon(ch["CNO"], ch["saline"])
        print(f"--- {L}, RE low {lowset}: change in end minus middle per mouse:\n{ch.round(1).to_string()}")
        print(f"    mean change saline {ch['saline'].mean():.1f}, DCZ {ch['DCZ'].mean():.1f}, CNO {ch['CNO'].mean():.1f}, RE low {ch['RE low'].mean():.1f}; "
              f"DCZ vs saline paired t p={t.pvalue:.3f}; CNO vs saline Wilcoxon p={w.pvalue:.3f}; |RE low| > |DCZ|: {abs(ch['RE low'].mean()) > abs(ch['DCZ'].mean())}")
pfc = {n: pd.concat([fields_df(s, n) for s in DR])["com"].dropna() for n in ["BeforeSaline", "AfterSaline", "BeforeDCZ", "AfterDCZ", "BeforeCNO", "AfterCNO", "Rewarded"]}
pfc["RE low"] = pd.concat([fields_df(s, "RElow_eng_L1") for s in DR])["com"].dropna()
print("Gaussian fits mean +/- SD (cm):", {n: tuple(np.round(stats.norm.fit(v), 1)) for n, v in pfc.items()}, "| within 10% of the paper's means:",
      {n: abs(stats.norm.fit(v)[0] - EXP4M[n]) <= 0.1 * EXP4M[n] for n, v in pfc.items()})

**Result: reproduced.** The change in end minus middle field percentage is 2.1 points after DCZ against 4.3 after saline (paired t p = 0.402, not significant), and the RE low change (-7.3 under L1, engaged laps) is larger in size; CNO against saline Wilcoxon p = 0.625. DCZ does not reduce the over-representation in absolute terms here; it rises less than after saline. All Gaussian means are within 10% of the paper's.

## Blocked claims

- supp2-decoder-stable-control and controls-no-change-over-time: the control mice that stayed in R for the whole session are not in the dandiset (its 14 files are the 10 extinction mice and the 4 DREADD mice).
- supp6-pupil-engagement: no pupil video or pupil traces are in the files.
- supp15-tdtomato-controls: the three DAT-Cre tdTomato mice are not in the dandiset; all four DREADD files are described as hM4D(Gi) mice.
- fig5-da-axon-ramps, fig5k-ramp-slope-max, fig5l-ramps-return-rr, supp20-novel-axons, supp21-heterogeneous-axons: the VTA axon imaging is not in the dandiset.
- supp14-reward-cells: the reward cell definition is "as described in ref. 24": a cell that "fired at the reward zone on the track (40 cm before reward) and around reward delivery (2 seconds before and after)". Neither the paper nor the claims file says what counts as firing there (a place field with its centre in the zone, a peak, a threshold on activity), and the criterion is not in the paper's code. Any threshold chosen here would be arbitrary, and the count of 43 cells is the claim's main number.

## Summary

| Claim | Outcome |
|---|---|
| session-lap-counts | reproduced |
| lick-persistence-ur | partially reproduced |
| fig1d-decoder-drop-ur | reproduced |
| fig1e-decoder-follows-licking | reproduced |
| fig1f-decoder-r2 | partially reproduced |
| fig1g-decoder-error-all-locations | reproduced |
| supp4-lickstop-definition-robust | reproduced |
| supp4-lapwise-r2 | partially reproduced |
| supp2-decoder-stable-control | blocked (data missing) |
| supp5-not-running-speed | partially reproduced |
| approach-ratio | not reproduced |
| engaged-lap-counts | partially reproduced |
| supp6b-decoder-engaged-laps | reproduced |
| disengaged-laps-uniform | reproduced |
| supp6-pupil-engagement | blocked (data missing) |
| fig2a-pv-cluster-transition | reproduced |
| fig2c-spatial-correlation-drop | reproduced |
| fig2d-remapping-all-locations | reproduced |
| fig2e-place-field-fates | not reproduced |
| fig3a-field-numbers-similar | not reproduced |
| fig3b-reliability | reproduced |
| fig3b-out-in-ratio | reproduced |
| fig3b-width-intensity | not reproduced |
| fig3c-degradation-all-locations | partially reproduced |
| supp11-all-mice-parameters | not reproduced |
| fig3d-reward-overrepresentation | partially reproduced |
| reward-zone-field-turnover | not reproduced |
| controls-no-change-over-time | blocked (data missing) |
| supp13-novel-environment | partially reproduced |
| supp14-reward-cells | blocked (methods underspecified) |
| supp16-dreadd-running-speed | partially reproduced |
| fig4f-dreadd-spatial-correlation | partially reproduced |
| supp18-saline-slow-laps | not reproduced |
| fig4h-dreadd-reliability-outin | partially reproduced |
| fig4m-dreadd-overrepresentation | reproduced |
| supp15-tdtomato-controls | blocked (data missing) |
| fig5-da-axon-ramps, fig5k, fig5l, supp20, supp21 | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's.

- The dandiset holds 10 of the 12 extinction mice (5 of the 6 with RR and a novel session) and 4 of the 6 DREADD mice (4 of 5 with saline). Counts of laps, fields and cells are over these mice.
- The files hold raw fluorescence, not the paper's dF/F. dF/F here uses the 8th percentile in a 15 s window, and significant transients start above 2 and end below 0.5 robust SD and last 0.5 s or more (ref. 75; assumption). Place field candidates are bins above 25% of the way from baseline to peak (ref. 75), and the bootstrap uses 100 circular shifts. The fields found are about twice as wide as the paper's.
- The DREADD files carry a constant offset of about 17000 counts, removed as the file minimum before dF/F (assumption).
- Position is in volts; the top of the range, where reward is delivered, is read as 200 cm. Lap times and speeds from it are longer and slower than the paper's (R laps 12.2 s against 7.34 s; DREADD lap speeds about 26 against 42 cm/s), which suggests that the scale, or the paper's definition of lap time, differs.
- The decoder treats the track as a circle by taking each prediction at its nearest copy of the true bin, as the criterion's "track treated as circular" says; without it R2 values are far lower (shown).
- The approach ratio is taken over the whole lap (teleport to teleport) for the engagement rule, the reading that gives 70% engaged laps and 3 mice with more than 12 engaged laps as the paper reports; the traversal-only reading is shown.
- The reward zone starts at the mean position of the first R lick after 20 cm (licks just after the teleport belong to the previous reward; assumption).
- The within-R reading S1 (control mice) of the spatial correlation could not be run.
- The saline split of Supplementary Fig. 18 uses percentiles within each mouse only.

What was found about the dataset.

- sub-014 has 6000 fewer behaviour rows than fluorescence frames; decoding shows the missing rows are at the start of BeforeCNO, so every later behaviour row is 6000 frames early. sub-NR6 has 8 extra behaviour rows and sub-CFC4 one row fewer, at the end.
- The DREADD fluorescence carries an undocumented offset of about 17000 counts.
- Conditions are not marked in the files except as frame counts in a free-text description; the position channel has no unit or scale; the files do not say which mice the paper's figures use.
- The control mice, the dark sessions, the pupil video, the tdTomato mice and the VTA axon imaging are not in the dandiset, and 2 extinction mice and 2 DREADD mice are missing.